# Updating Data in Mongoose

There are two broad approaches, and the choice between them matters more than the individual method names:

1. **Atomic updates** — `updateOne`, `findOneAndUpdate` and friends send an update instruction to MongoDB. One round-trip, no race conditions, but no `save` middleware and no validation unless you ask for it.
2. **Read-modify-write** — fetch a document, mutate it in JS, call `save()`. Full validation and middleware, but two round-trips and a window where another process can change the data underneath you.

## Common update methods

| Method | Returns | Runs `save` hooks |
|---|---|---|
| `findOneAndUpdate(filter, update, opts)` | The document — **old version by default** | No |
| `findByIdAndUpdate(id, update, opts)` | Same, matched by `_id` | No |
| `updateOne(filter, update, opts)` | Result metadata, not the document | No |
| `updateMany(filter, update, opts)` | Result metadata for all matches | No |
| `replaceOne(filter, doc)` | Metadata; replaces the whole document | No |
| `doc.save()` | The saved document | Yes |

Metadata shape returned by `updateOne` / `updateMany`:

```javascript
{ acknowledged: true, matchedCount: 1, modifiedCount: 1, upsertedId: null, upsertedCount: 0 }
```

`matchedCount: 0` means the filter found nothing. `matchedCount: 1, modifiedCount: 0` means it found the document but the new value was identical to the old one — a useful distinction when deciding between a 404 and a no-op.

## The `new: true` option

`findOneAndUpdate` returns the document **as it was before the update** by default. This surprises everyone once.

```javascript
const movie = await Movie.findOneAndUpdate(
  { title: 'Amelie' },
  { score: 9.0 },
  { new: true, runValidators: true }
);
```

`{ new: true }` is equivalent to `{ returnDocument: 'after' }`. Add it to essentially every `findOneAndUpdate` call you write, unless you specifically need the prior state.

## Update operators

Mongoose wraps top-level keys in `$set` for you, so these two are the same:

```javascript
await Movie.updateOne({ _id: id }, { score: 9.0 });
await Movie.updateOne({ _id: id }, { $set: { score: 9.0 } });
```

But the explicit operators do things plain keys can't:

| Operator | Effect |
|---|---|
| `$set` | Set a field's value |
| `$unset` | Remove a field entirely |
| `$inc` | Increment/decrement a number atomically |
| `$mul` | Multiply |
| `$min` / `$max` | Update only if the new value is lower/higher |
| `$rename` | Rename a field |
| `$currentDate` | Set to the server's current time |
| `$push` / `$addToSet` | Append to an array (`$addToSet` skips duplicates) |
| `$pull` / `$pop` | Remove from an array |

```javascript
await Post.updateOne({ _id: id }, { $inc: { views: 1 } });
await Post.updateOne({ _id: id }, { $push: { tags: 'mongoose' } });
await Post.updateOne({ _id: id }, { $push: { comments: { $each: [c1, c2], $slice: -10 } } });
await User.updateOne({ _id: id }, { $unset: { tempToken: '' } });
```

`$inc` is the canonical example of why atomic updates matter — `doc.views += 1; await doc.save()` loses increments under concurrency, `$inc` never does.

Updating an element inside an array uses the positional operator:

```javascript
await Order.updateOne(
  { _id: id, 'items.sku': 'ABC' },
  { $set: { 'items.$.qty': 5 } }
);
```

## Validation on updates

Schema validators **do not run** on `updateOne`, `updateMany`, `findOneAndUpdate` or `findByIdAndUpdate` unless you opt in:

```javascript
await Movie.findByIdAndUpdate(id, { score: 99 }, { new: true, runValidators: true });
```

Even with the flag, update validators behave differently from document validators:

- Only fields actually present in the update are validated. A `required` field you didn't touch isn't re-checked.
- In a custom validator, `this` is the **query**, not the document — so validators that read sibling fields break. Pass `context: 'query'` and use `this.getUpdate()` if you need them.
- `unique` still isn't a validator; duplicates come back as `E11000`.

## Upsert

Insert the document if the filter matches nothing:

```javascript
await Settings.findOneAndUpdate(
  { userId },
  { $set: { theme: 'dark' } },
  { new: true, upsert: true, setDefaultsOnInsert: true }
);
```

`setDefaultsOnInsert` applies schema defaults to the newly created document; without it you get only the fields in the filter and update. Note that concurrent upserts on the same filter can produce a duplicate-key error unless the filter fields carry a unique index.

## Middleware

`save` hooks don't fire on any of the atomic update methods. Password hashing in `pre('save')` is silently skipped by `findByIdAndUpdate` — the classic version of this bug writes a plaintext password to the database.

Query middleware does fire, and `this` is the query:

```javascript
schema.pre('findOneAndUpdate', function (next) {
  this.set({ updatedBy: 'system' });
  next();
});
```

`timestamps: true` does update `updatedAt` on these methods, so that much is handled for you.

## The `save()` approach

```javascript
const movie = await Movie.findById(id);
if (!movie) return res.status(404).send('Not found');

movie.score = 9.0;
movie.tags.push('classic');
await movie.save();   // full validation + save hooks
```

Use this when the new value depends on the old one in a way operators can't express, or when you need document middleware. Accept that it's two round-trips and not atomic — if that matters, enable optimistic concurrency with `{ optimisticConcurrency: true }` in the schema options, which makes `save()` throw a `VersionError` if the document changed since you read it.

## Choosing a method

| Situation | Use |
|---|---|
| Need the updated document back | `findByIdAndUpdate(..., { new: true })` |
| Fire-and-forget field change | `updateOne` |
| Counters, array pushes, anything concurrent | `updateOne` with `$inc` / `$push` |
| Bulk change across many documents | `updateMany` |
| Logic in `pre('save')` must run | fetch → mutate → `save()` |
| Mixed inserts and updates in one trip | `bulkWrite` |

## Gotchas

- **Forgetting `new: true`** — you get the stale document and think the update failed.
- **Forgetting `runValidators: true`** — invalid data lands in the database without complaint.
- **`replaceOne` wipes unspecified fields.** `updateOne` merges; `replaceOne` substitutes the entire document.
- **Invalid ObjectId throws a `CastError`** on `findByIdAndUpdate`, same as `findById`.
- **Never spread `req.body` straight into an update.** A client can send `{ role: 'admin' }` or operator keys. Whitelist the fields, or use `{ sanitizeFilter: true }` on the filter side.
- **`strict` applies to updates too** — keys not in the schema are dropped silently rather than raising an error.
- **Empty updates are a no-op**, not an error: `updateOne(filter, {})` returns `modifiedCount: 0`.

## Sources

- [findOneAndUpdate tutorial](https://mongoosejs.com/docs/tutorials/findoneandupdate.html)
- [Documents](https://mongoosejs.com/docs/documents.html)
- [Validation — update validators](https://mongoosejs.com/docs/validation.html)
- [MongoDB update operators](https://www.mongodb.com/docs/manual/reference/operator/update/)